# IPA — distance method vs Kneedle (self-contained cells)

Three independent cells, no curve fitting anywhere (neither elbow needs it):

1. **Distance method** — elbow = data point farthest below the start→end diagonal;
   end anchor = mean of the last `DIST_TAIL_N = 20` CE points. Complete code + plot.
2. **Kneedle** — `kneed.KneeLocator`; bottom anchor set by `KNEEDLE_TAIL_N`:
   `20` = mean of last 20 points, `None` = standard `min(y)`. Complete code + plot.
3. **Comparison** — overlays both IPA-vs-P% curves per batch size and prints where
   the chosen elbows (BN_learned) differ.

Shared conventions: step-artifact truncation (`BN_STEP_MIN=100`, `STEP_THRESH=0.01`),
`CE_o = ln(10)`, `IPA = |CE_o − CE_learned| / BN_learned`. Requires `pip install kneed`.

In [4]:
# ============================================================================
# Cell 1 — DISTANCE METHOD (complete, self-contained)
#
# Elbow = the data point with maximum perpendicular distance BELOW the
# diagonal from (BN[0], CE[0]) to (BN[-1], A), where the end anchor
# A = mean of the last DIST_TAIL_N CE points (empirical floor).
# IPA = |CE_o - CE_learned| / BN_learned
# No curve fitting needed. Produces the distance-method IPA vs P% plot.
# ============================================================================
import os, glob, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BN_STEP_MIN = 100     # don't check for step artifacts before this BN
STEP_THRESH = 0.01    # |CE[i]-CE[i-1]| threshold for step artifact
DIST_TAIL_N = 20      # end anchor = mean of the last N CE points

BASE_DIR = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\prune_layers_ALL"
OUT_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step"
BATCH_SIZES = [64, 1024, 60000]
CE_o = np.log(10)     # max CE for 10-class problem, ~2.302585
BS_COLOR = {64: "#1f77b4", 1024: "#d62728", 60000: "#2ca02c"}

PRUNING_LEVELS = sorted(float(re.search(r"p-percentage_([\d.]+)", d).group(1))
                        for d in glob.glob(os.path.join(BASE_DIR, "p-percentage_*")))
print(f"Found {len(PRUNING_LEVELS)} pruning levels")


def load_curve(p, bs):
    """Averaged CE curve for (p, bs), truncated at the step artifact.
    Returns (BN, CE) arrays or None."""
    f = os.path.join(BASE_DIR, f"p-percentage_{p}", f"batch_size_{bs}",
                     f"averaged_runs_p_{p}_bs_{bs}.csv")
    if not os.path.exists(f):
        return None
    df = pd.read_csv(f)
    df.columns = df.columns.str.strip()
    ce_col = next((c for c in df.columns if c in ("Avg_CE_Test", "Avg_CE_test")), None)
    bn_col = next((c for c in df.columns if "Batch" in c), None)
    if ce_col is None or bn_col is None:
        return None
    df = df.dropna(subset=[ce_col, bn_col])
    bns = df[bn_col].values.astype(float)
    ces = df[ce_col].values.astype(float)
    cutoff_BN = float(bns[-1])                       # same convention as main notebook
    for i in range(1, len(bns)):
        if bns[i] >= BN_STEP_MIN and abs(ces[i] - ces[i - 1]) > STEP_THRESH:
            cutoff_BN = float(bns[i])
            break
    m = bns < cutoff_BN
    return bns[m], ces[m]


def ipa_distance(BN, CE, tail_n=DIST_TAIL_N):
    """Perpendicular-distance elbow -> (BN_learned, CE_learned, IPA)."""
    BN = np.asarray(BN, float)
    CE = np.asarray(CE, float)
    if len(BN) < 2:
        return np.nan, np.nan, np.nan
    k = int(min(tail_n, len(CE)))
    A = float(np.mean(CE[-k:]))                      # end anchor = tail average
    BN_range = BN[-1] - BN[0]
    CE_range = CE[0] - A
    if BN_range <= 0 or CE_range <= 1e-10:           # degenerate flat curve (P=100%)
        return np.nan, np.nan, np.nan
    x = (BN - BN[0]) / BN_range                      # normalize to [0,1]
    y = (CE - A) / CE_range
    i = int(np.argmax(1.0 - (x + y)))                # max distance BELOW x+y=1 diagonal
    if BN[i] <= 0:
        return np.nan, np.nan, np.nan
    return float(BN[i]), float(CE[i]), abs(CE_o - CE[i]) / BN[i]


rows = []
for p in PRUNING_LEVELS:
    row = {"P%": p * 100}
    for bs in BATCH_SIZES:
        curve = load_curve(p, bs)
        bn_l, ce_l, ipa = ipa_distance(*curve) if curve is not None else (np.nan,)*3
        row[f"BN_learned_{bs}"] = bn_l
        row[f"IPA_Avg_{bs}"]    = ipa
    rows.append(row)
dist_summary_df = pd.DataFrame(rows)

csv_path = os.path.join(OUT_DIR, f"ipa_summary_distance_tail{DIST_TAIL_N}.csv")
dist_summary_df.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")
print(dist_summary_df[["P%"] + [f"IPA_Avg_{b}" for b in BATCH_SIZES]].to_string(index=False))

plt.rcParams.update({"font.size": 13})
fig, ax = plt.subplots(figsize=(9, 5.5))
for bs in BATCH_SIZES:
    sub = dist_summary_df.dropna(subset=[f"IPA_Avg_{bs}"])
    ax.plot(sub["P%"], sub[f"IPA_Avg_{bs}"], "o-", color=BS_COLOR[bs], ms=5, lw=2, label=f"BS={bs}")
ax.set_xlabel("Pruning Percentage (%)")
ax.set_ylabel("IPA")
ax.set_title(f"IPA vs Pruning — distance method (end anchor = mean of last {DIST_TAIL_N})")
ax.grid(True, alpha=0.3)
ax.legend(frameon=False)
png_path = os.path.join(OUT_DIR, f"ipa_plot_distance_tail{DIST_TAIL_N}.png")
plt.tight_layout(); plt.savefig(png_path, dpi=150, bbox_inches="tight"); plt.close(fig)
print(f"Saved: {png_path}")


Found 19 pruning levels
Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\ipa_summary_distance_tail20.csv
   P%  IPA_Avg_64  IPA_Avg_1024  IPA_Avg_60000
  0.0    0.057926      0.067119       0.076981
 10.0    0.060861      0.062432       0.073748
 20.0    0.052078      0.065791       0.072946
 30.0    0.048767      0.055950       0.069466
 40.0    0.048921      0.054963       0.061973
 50.0    0.040618      0.047364       0.057097
 60.0    0.032051      0.042872       0.047341
 70.0    0.026562      0.036108       0.038163
 80.0    0.021318      0.027689       0.030385
 82.0    0.018689      0.025826       0.027507
 84.0    0.017850      0.024001       0.025382
 86.0    0.016741      0.021854       0.023611
 88.0    0.015208      0.020305       0.022110
 90.0    0.013781      0.018431       0.019321
 92.0    0.012057      0.015841       0.016624
 94.0    0.010137      0.013527       0.014321
 96.0    0.008081    

In [9]:
# ============================================================================
# Cell 2 — KNEEDLE METHOD (complete, self-contained)
#
# Elbow via kneed.KneeLocator (curve='convex', direction='decreasing', S=1.0).
# Bottom normalization anchor is configurable:
#   KNEEDLE_TAIL_N = 20    -> anchor = mean of the last 20 CE points
#                             (curve clipped at that floor before detection)
#   KNEEDLE_TAIL_N = None  -> standard Kneedle: anchor = min(y) (single point)
# IPA = |CE_o - CE_learned| / BN_learned
# Produces the Kneedle IPA vs P% plot.
# ============================================================================
import os, glob, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from kneed import KneeLocator

BN_STEP_MIN    = 100
STEP_THRESH    = 0.01
KNEEDLE_S      = 1.0
KNEEDLE_TAIL_N = None      # int = tail-average anchor; None = standard min(y)

BASE_DIR = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\prune_layers_ALL"
OUT_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step"
BATCH_SIZES = [64, 1024, 60000]
CE_o = np.log(10)
BS_COLOR = {64: "#1f77b4", 1024: "#d62728", 60000: "#2ca02c"}
ANCHOR_TAG = f"tail{KNEEDLE_TAIL_N}" if KNEEDLE_TAIL_N is not None else "min"

PRUNING_LEVELS = sorted(float(re.search(r"p-percentage_([\d.]+)", d).group(1))
                        for d in glob.glob(os.path.join(BASE_DIR, "p-percentage_*")))
print(f"Found {len(PRUNING_LEVELS)} pruning levels;  Kneedle anchor: {ANCHOR_TAG}")


def load_curve(p, bs):
    """Averaged CE curve for (p, bs), truncated at the step artifact.
    Returns (BN, CE) arrays or None."""
    f = os.path.join(BASE_DIR, f"p-percentage_{p}", f"batch_size_{bs}",
                     f"averaged_runs_p_{p}_bs_{bs}.csv")
    if not os.path.exists(f):
        return None
    df = pd.read_csv(f)
    df.columns = df.columns.str.strip()
    ce_col = next((c for c in df.columns if c in ("Avg_CE_Test", "Avg_CE_test")), None)
    bn_col = next((c for c in df.columns if "Batch" in c), None)
    if ce_col is None or bn_col is None:
        return None
    df = df.dropna(subset=[ce_col, bn_col])
    bns = df[bn_col].values.astype(float)
    ces = df[ce_col].values.astype(float)
    cutoff_BN = float(bns[-1])
    for i in range(1, len(bns)):
        if bns[i] >= BN_STEP_MIN and abs(ces[i] - ces[i - 1]) > STEP_THRESH:
            cutoff_BN = float(bns[i])
            break
    m = bns < cutoff_BN
    return bns[m], ces[m]


def ipa_kneedle(BN, CE, tail_n=KNEEDLE_TAIL_N, S=KNEEDLE_S):
    """Kneedle elbow -> (BN_learned, CE_learned, IPA).
    CE_learned is always read from the ORIGINAL (unclipped) data."""
    BN = np.asarray(BN, float)
    CE = np.asarray(CE, float)
    if len(BN) < 3 or np.ptp(CE) <= 1e-10:           # degenerate flat curve (P=100%)
        return np.nan, np.nan, np.nan
    if tail_n is not None:
        k = int(min(tail_n, len(CE)))
        floor = float(np.mean(CE[-k:]))
        if CE[0] - floor <= 1e-10:
            return np.nan, np.nan, np.nan
        y_for_knee = np.clip(CE, floor, None)        # min(y_for_knee) == floor
    else:
        y_for_knee = CE                              # standard: anchor = min(CE)
    try:
        kl = KneeLocator(BN, y_for_knee, curve="convex", direction="decreasing", S=S)
    except Exception:
        return np.nan, np.nan, np.nan
    if kl.knee is None:
        return np.nan, np.nan, np.nan
    i = int(np.argmin(np.abs(BN - kl.knee)))
    if BN[i] <= 0:
        return np.nan, np.nan, np.nan
    return float(BN[i]), float(CE[i]), abs(CE_o - CE[i]) / BN[i]


rows = []
for p in PRUNING_LEVELS:
    row = {"P%": p * 100}
    for bs in BATCH_SIZES:
        curve = load_curve(p, bs)
        bn_l, ce_l, ipa = ipa_kneedle(*curve) if curve is not None else (np.nan,)*3
        row[f"BN_learned_{bs}"] = bn_l
        row[f"IPA_Avg_{bs}"]    = ipa
    rows.append(row)
kneedle_summary_df = pd.DataFrame(rows)

csv_path = os.path.join(OUT_DIR, f"ipa_summary_kneedle_{ANCHOR_TAG}.csv")
kneedle_summary_df.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")
print(kneedle_summary_df[["P%"] + [f"IPA_Avg_{b}" for b in BATCH_SIZES]].to_string(index=False))

plt.rcParams.update({"font.size": 13})
fig, ax = plt.subplots(figsize=(9, 5.5))
for bs in BATCH_SIZES:
    sub = kneedle_summary_df.dropna(subset=[f"IPA_Avg_{bs}"])
    ax.plot(sub["P%"], sub[f"IPA_Avg_{bs}"], "s--", color=BS_COLOR[bs], ms=5, lw=2, label=f"BS={bs}")
ax.set_xlabel("Pruning Percentage (%)")
ax.set_ylabel("IPA")
anchor_desc = (f"mean of last {KNEEDLE_TAIL_N}" if KNEEDLE_TAIL_N is not None else "min(y), standard")
ax.set_title(f"IPA vs Pruning — Kneedle (S={KNEEDLE_S}, anchor = {anchor_desc})")
ax.grid(True, alpha=0.3)
ax.legend(frameon=False)
png_path = os.path.join(OUT_DIR, f"ipa_plot_kneedle_{ANCHOR_TAG}.png")
plt.tight_layout(); plt.savefig(png_path, dpi=150, bbox_inches="tight"); plt.close(fig)
print(f"Saved: {png_path}")


Found 19 pruning levels;  Kneedle anchor: min
Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\ipa_summary_kneedle_min.csv
   P%  IPA_Avg_64  IPA_Avg_1024  IPA_Avg_60000
  0.0    0.054735      0.067119       0.076981
 10.0    0.047737      0.062432       0.073748
 20.0    0.048365      0.058271       0.072946
 30.0    0.048767      0.055950       0.069466
 40.0    0.040060      0.056493       0.061973
 50.0    0.035555      0.047364       0.057097
 60.0    0.032051      0.041137       0.047341
 70.0    0.025533      0.034859       0.038163
 80.0    0.020862      0.026574       0.030385
 82.0    0.018689      0.025826       0.027507
 84.0    0.017850      0.024001       0.025712
 86.0    0.016741      0.021854       0.023611
 88.0    0.014170      0.019878       0.022110
 90.0    0.013475      0.018431       0.019321
 92.0    0.011734      0.015982       0.016624
 94.0    0.010137      0.013527       0.014321
 96

In [10]:
# ============================================================================
# Cell 3 — COMPARISON: distance method vs Kneedle
#
# Uses `dist_summary_df` (Cell 1) and `kneedle_summary_df` (Cell 2) if they
# exist in memory; otherwise reloads them from the CSVs written by those cells.
# One panel per batch size + a BN_learned difference printout.
# ============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step"
BATCH_SIZES = [64, 1024, 60000]
BS_COLOR = {64: "#1f77b4", 1024: "#d62728", 60000: "#2ca02c"}

if "dist_summary_df" not in dir():
    dist_summary_df = pd.read_csv(os.path.join(OUT_DIR, "ipa_summary_distance_tail20.csv"))
    print("(reloaded distance summary from CSV)")
if "kneedle_summary_df" not in dir():
    kneedle_summary_df = pd.read_csv(os.path.join(OUT_DIR, "ipa_summary_kneedle_tail20.csv"))
    print("(reloaded kneedle summary from CSV)")
dist_lbl = f"distance (anchor: mean last {DIST_TAIL_N})"    if "DIST_TAIL_N"    in dir() else "distance"
kn_lbl   = (f"Kneedle (anchor: mean last {KNEEDLE_TAIL_N})" if KNEEDLE_TAIL_N is not None
            else "Kneedle (anchor: min(y))")                if "KNEEDLE_TAIL_N" in dir() else "Kneedle"

plt.rcParams.update({"font.size": 13})
fig, axes = plt.subplots(1, len(BATCH_SIZES), figsize=(6 * len(BATCH_SIZES), 5), sharex=True)
if len(BATCH_SIZES) == 1:
    axes = [axes]
for ax, bs in zip(axes, BATCH_SIZES):
    col = f"IPA_Avg_{bs}"
    d = dist_summary_df.dropna(subset=[col])
    k = kneedle_summary_df.dropna(subset=[col])
    ax.plot(d["P%"], d[col], "o-",  color=BS_COLOR[bs], ms=5, lw=2,   label=dist_lbl)
    ax.plot(k["P%"], k[col], "s--", color="#555555",    ms=5, lw=1.6, label=kn_lbl)
    ax.set_title(f"BS={bs}")
    ax.set_xlabel("Pruning Percentage (%)")
    ax.grid(True, alpha=0.3)
    ax.legend(frameon=False, fontsize=9)
axes[0].set_ylabel("IPA")
fig.suptitle("IPA vs Pruning — distance method vs Kneedle", fontsize=13)
png_path = os.path.join(OUT_DIR, "ipa_plot_compare_distance_vs_kneedle.png")
plt.tight_layout(); plt.savefig(png_path, dpi=150, bbox_inches="tight"); plt.close(fig)
print(f"Saved: {png_path}")

# Where do the two methods pick different elbows?
print("\nBN_learned differences (kneedle - distance):")
for bs in BATCH_SIZES:
    c = f"BN_learned_{bs}"
    if c in dist_summary_df.columns and c in kneedle_summary_df.columns:
        diff = kneedle_summary_df[c] - dist_summary_df[c]
        changed = dist_summary_df["P%"][diff.fillna(0) != 0]
        with np.printoptions(legacy="1.25"):
            print(f"  BS={bs}: {len(changed)} of {diff.notna().sum()} differ  "
                  f"P%={list(changed.values)}  dBN={list(diff[diff.fillna(0) != 0].values)}")


Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\ipa_plot_compare_distance_vs_kneedle.png

BN_learned differences (kneedle - distance):
  BS=64: 11 of 18 differ  P%=[0.0, 10.0, 20.0, 40.0, 50.0, 70.0, 80.0, 88.0, 90.0, 92.0, 96.0]  dBN=[2.0, 9.0, 3.0, 9.0, 7.0, 3.0, 2.0, 9.0, 3.0, 4.0, 3.0]
  BS=1024: 7 of 18 differ  P%=[20.0, 40.0, 60.0, 70.0, 80.0, 88.0, 92.0]  dBN=[4.0, -1.0, 2.0, 2.0, 3.0, 2.0, -1.0]
  BS=60000: 2 of 18 differ  P%=[84.0, 98.0]  dBN=[-1.0, -1.0]
